## Take clipped data and prep for export
Author: Lara Spearman <br>
Date: 01/09/2025 <br>
Purpose: Takes csv's outputted form clipping analysis.ipynb and adds ranking for each dataset

In [10]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import json

In [11]:
# Get current working directory
cwd = Path.cwd()
location_data = "C:/Users/LaraS/Wiltshire Wildlife Trust/WSBRC Data - Documents/Services/Neighbourhood_Planning/_NeigbourhoodPlansPython/NeigbourhoodPlansPython.gdb"

In [12]:
# Import file information to read in
with open('file_info_gdb.json') as f:
    file_info_shp = json.load(f)
    print(file_info_shp)

{'Landscape Character Areas': {'file_name': '', 'source': 'Wiltshire Council', 'group': 'LCA', 'group_field': 'Type_name', 'groupColumnName': 'Area', 'groupColumnValue': False, 'unit': 'Ha', 'unit_type': 'AREA', 'about': 'Landscape Character Areas represent an area of distinct and recognisable character at the national scale. Their boundaries follow natural lines in the landscape, not county or district boundaries. This makes them a good framework for decision-making and planning for future change. ', 'layer_name': 'LCA_County'}, 'Bedrock': {'file_name': '', 'source': 'British Geological Survey', 'group': 'Geology', 'group_field': 'LEX_RCS_D', 'groupColumnName': 'Bedrock', 'groupColumnValue': False, 'unit': 'Ha', 'unit_type': 'AREA', 'about': 'test', 'layer_name': 'Bedrock'}, 'Faultlines': {'file_name': '', 'source': 'British Geological Survey', 'group': 'Geology', 'group_field': 'CATEGORY', 'groupColumnName': 'Faultline', 'groupColumnValue': False, 'unit': 'Metres', 'unit_type': 'LENG

In [13]:
# Read in parish council boundaries
gdf = gpd.read_file(location_data, layer= "OS_BL_Parish_Wilts")
gdf = gdf.to_crs("EPSG:27700")
gdf['AreaPC'] = gdf['Hectares']

In [14]:
# Interim step, read in all csv's and combine
df_all_shp = pd.DataFrame({'Name': [], 'value': [],'groupColumnValue': [],'groupColumnName': [],'mapGroup': [], "unitName":[],"unit":[],"datasetName":[], "source":[]})


for file in file_info_shp.keys():
    df = pd.read_csv(f"{cwd}/data/clippedData/{file}_v1.csv")
    df_all_shp = pd.concat([df_all_shp, df])

In [15]:
# Add area of PC to datasets
df_all_shp = pd.merge(df_all_shp, gdf[['Name', 'AreaPC']], how = 'left', on = "Name")
# Add value percent column
def add_percent(row):
    if row['unitName'].lower() == "area":
        val =  (100*row['value'])/row['AreaPC']
    else:
        val =   ""
    return val
    
df_all_shp['valuePercent'] = df_all_shp.apply(add_percent, axis = 1)

In [16]:
# Get rank for each PC on how it compares for each dataset to other PCs
df_all_ranked = pd.DataFrame({'Name': [], 'value': [],'valuePercent': [],'groupColumnValue': [],'groupColumnName': [],'mapGroup': [],'mapName': [], 'rank':[], 'rankPercent':[]})

for dataset in df_all_shp.datasetName.unique():
    
    df_dataset = df_all_shp[df_all_shp['datasetName']==dataset]
   # Check if groupColumnValue is null (or na), which means no groups to loop over
    if pd.isnull(df_dataset.groupColumnValue.unique()[0])==True:
        # Sort PC's by value
        df_rank_sorted = df_dataset.sort_values(by = "value", ascending=False)
        # Add rank column which ranks PC's
        df_rank_sorted['rank'] = list(range(1, df_rank_sorted.shape[0]+1))

        # Sort PC's by percentage
        df_rank_percent_sorted = df_rank_sorted.sort_values(by = "valuePercent", ascending=False)
        # Rank PC's by percent
        df_rank_percent_sorted['rankPercent'] = list(range(1, df_rank_percent_sorted.shape[0]+1))
        
        df_all_ranked = pd.concat([df_all_ranked, df_rank_percent_sorted])

    else:
        # If dataset has groupings, need to loop through
        for group in df_dataset.groupColumnValue.unique():
            # Sort PC's by value
            df_rank= df_dataset[df_dataset.groupColumnValue == group]
            df_rank_sorted = df_rank.sort_values(by = "value", ascending=False)
            # Add rank column which ranks PC's
            df_rank_sorted['rank'] = list(range(1, df_rank_sorted.shape[0]+1))

             # Sort PC's by percentage
            df_rank_percent_sorted = df_rank_sorted.sort_values(by = "valuePercent", ascending=False)
            # Rank PC's by percent
            df_rank_percent_sorted['rankPercent'] = list(range(1, df_rank_percent_sorted.shape[0]+1))
            
            df_all_ranked = pd.concat([df_all_ranked, df_rank_percent_sorted])

In [17]:
df_all_ranked['unit'] = df_all_ranked['unit'].replace("HECTARES", "Ha")
df_all_ranked['unit'] = df_all_ranked['unit'].replace("Area (ha)", "Ha")
df_all_ranked['unit'] = df_all_ranked['unit'].replace("METERS", "Metres")
df_all_ranked['unit'] = df_all_ranked['unit'].replace("Count", " ")

In [18]:
df_all_ranked.to_csv(f"{cwd}/data/outputClippedData.csv", index=False)